In [7]:
import numpy as np 
import pandas as pd
import json
import sqlite3

In [15]:
df=pd.read_csv('orders.csv')

In [16]:
df.head(5)

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [13]:
df2 = pd.read_json("users.json")

In [14]:
df2.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [ ]:

conn = sqlite3.connect("restaurants.db")
cursor = conn.cursor()

with open("restaurants.sql", "r") as f:
    sql_script = f.read()

cursor.executescript(sql_script)   
conn.commit()



In [ ]:

df3 = pd.read_sql_query("SELECT * FROM restaurants;", conn)

df3.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [18]:
j1=pd.merge(df, df2, on="user_id", how="left")

In [20]:
j1.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [21]:
j2=pd.merge(j1, df3, on="restaurant_id", how='left')

In [52]:
j2.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           10000 non-null  int64         
 1   user_id            10000 non-null  int64         
 2   restaurant_id      10000 non-null  int64         
 3   order_date         10000 non-null  datetime64[ns]
 4   total_amount       10000 non-null  float64       
 5   restaurant_name_x  10000 non-null  object        
 6   name               10000 non-null  object        
 7   city               10000 non-null  object        
 8   membership         10000 non-null  object        
 9   restaurant_name_y  10000 non-null  object        
 10  cuisine            10000 non-null  object        
 11  rating             10000 non-null  float64       
 12  rating_range       10000 non-null  category      
 13  quarter            10000 non-null  int64         
dtypes: cate

In [24]:
gold_df=j2[j2['membership']== 'Gold']
city_revenue=gold_df.groupby('city')['total_amount'].sum().reset_index()

In [44]:
total_gold_orders = len(gold_df)
print("Total orders placed by Gold members:", total_gold_orders)

Total orders placed by Gold members: 4987


In [25]:
top_city = city_revenue.loc[city_revenue["total_amount"].idxmax()]

In [26]:
print(top_city)

city               Chennai
total_amount    1080909.79
Name: 1, dtype: object


In [27]:
avg_order_value = j2.groupby("cuisine")["total_amount"].mean().reset_index()
top_cuisine = avg_order_value.loc[avg_order_value["total_amount"].idxmax()]

In [28]:
print(top_cuisine)

cuisine            Mexican
total_amount    808.021344
Name: 3, dtype: object


In [ ]:

user_spend = j2.groupby("user_id")["total_amount"].sum().reset_index()

bins = [0, 500, 1000, 2000, float("inf")]
labels = ["< 500", "500 – 1000", "1000 – 2000", "> 2000"]

user_spend["spend_category"] = pd.cut(user_spend["total_amount"], bins=bins, labels=labels)

category_counts = user_spend["spend_category"].value_counts().sort_index()

print(category_counts)


< 500           114
500 – 1000      225
1000 – 2000     699
> 2000         1845
Name: spend_category, dtype: int64


In [31]:
percentages = (category_counts / category_counts.sum()) * 100
print(percentages)


< 500           3.954214
500 – 1000      7.804370
1000 – 2000    24.245578
> 2000         63.995838
Name: spend_category, dtype: float64


In [ ]:

bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0 - 3.5", "3.6 - 4.0", "4.1 - 4.5", "4.6 - 5.0"]

j2["rating_range"] = pd.cut(j2["rating"], bins=bins, labels=labels, include_lowest=True)

rating_revenue = j2.groupby("rating_range")["total_amount"].sum().reset_index()

top_range = rating_revenue.loc[rating_revenue["total_amount"].idxmax()]

print("Rating range with highest total revenue:")
print(top_range)


Rating range with highest total revenue:
rating_range     4.6 - 5.0
total_amount    2197030.75
Name: 3, dtype: object


In [ ]:

cuisine_stats = j2.groupby("cuisine").agg(
    distinct_restaurants=("restaurant_name_y", "nunique"),
    total_revenue=("total_amount", "sum")
).reset_index()

cuisine_stats_sorted = cuisine_stats.sort_values(by="distinct_restaurants")

print(cuisine_stats_sorted)


   cuisine  distinct_restaurants  total_revenue
0  Chinese                   120     1930504.65
1   Indian                   126     1971412.58
2  Italian                   126     2024203.80
3  Mexican                   128     2085503.09


In [ ]:

total_orders = len(j2)

gold_orders = len(j2[j2["membership"] == "Gold"])

percentage = round((gold_orders / total_orders) * 100)

print(f"{percentage}% of total orders were placed by Gold members")


50% of total orders were placed by Gold members


In [ ]:

restaurant_stats = j2.groupby("restaurant_name_y").agg(
    total_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
).reset_index()

filtered = restaurant_stats[restaurant_stats["total_orders"] < 20]

top_restaurant = filtered.loc[filtered["avg_order_value"].idxmax()]

print("Restaurant with highest average order value but < 20 orders:")
print(top_restaurant)


Restaurant with highest average order value but < 20 orders:
restaurant_name_y    Restaurant_294
total_orders                     13
avg_order_value         1040.222308
Name: 216, dtype: object


In [ ]:

options = ["Grand Cafe Punjabi", "Grand Restaurant South Indian", 
           "Ruchi Mess Multicuisine", "Ruchi Foods Chinese"]

filtered_options = j2[j2["restaurant_name_x"].isin(options)]

option_stats = filtered_options.groupby("restaurant_name_x").agg(
    total_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
).reset_index()

option_stats = option_stats[option_stats["total_orders"] < 20]

top_option = option_stats.loc[option_stats["avg_order_value"].idxmax()]

print(top_option)


restaurant_name_x    Ruchi Foods Chinese
total_orders                          19
avg_order_value               686.603158
Name: 2, dtype: object


In [ ]:

combo_revenue = j2.groupby(["membership", "cuisine"])["total_amount"].sum().reset_index()

top_combo = combo_revenue.loc[combo_revenue["total_amount"].idxmax()]

print("Combination contributing highest revenue:")
print(top_combo)


Combination contributing highest revenue:
membership        Regular
cuisine           Mexican
total_amount    1072943.3
Name: 7, dtype: object


In [ ]:

combos = [
    ("Gold", "Indian"),
    ("Gold", "Italian"),
    ("Regular", "Indian"),
    ("Regular", "Chinese")
]

filtered_combos = j2[j2[["membership", "cuisine"]].apply(tuple, axis=1).isin(combos)]

combo_stats = filtered_combos.groupby(["membership", "cuisine"]).agg(
    total_revenue=("total_amount", "sum")
).reset_index()

top_combo = combo_stats.loc[combo_stats["total_revenue"].idxmax()]
print(top_combo)


membership             Gold
cuisine             Italian
total_revenue    1005779.05
Name: 1, dtype: object


In [ ]:

j2["order_date"] = pd.to_datetime(j2["order_date"], format="%d-%m-%Y")

j2["quarter"] = j2["order_date"].dt.quarter

quarter_revenue = j2.groupby("quarter")["total_amount"].sum().reset_index()

top_quarter = quarter_revenue.loc[quarter_revenue["total_amount"].idxmax()]

print("Quarter with highest total revenue:")
print(top_quarter)


Quarter with highest total revenue:
quarter               3.0
total_amount    2037385.1
Name: 2, dtype: float64


In [ ]:

hyd_orders = j2[j2["city"] == "Hyderabad"]
hyd_revenue = hyd_orders["total_amount"].sum()

hyd_revenue_rounded = round(hyd_revenue)

print("Total revenue from Hyderabad city:", hyd_revenue_rounded)


Total revenue from Hyderabad city: 1889367


In [ ]:


distinct_users = j2["user_id"].nunique()

print("Number of distinct users who placed at least one order:", distinct_users)


Number of distinct users who placed at least one order: 2883


In [ ]:

gold_orders = j2[j2["membership"] == "Gold"]

avg_order_value_gold = round(gold_orders["total_amount"].mean(), 2)

print("Average order value for Gold members:", avg_order_value_gold)


Average order value for Gold members: 797.15


In [ ]:

high_rating_orders = j2[j2["rating"] >= 4.5]

total_high_rating_orders = len(high_rating_orders)

print("Number of orders for restaurants with rating ≥ 4.5:", total_high_rating_orders)


Number of orders for restaurants with rating ≥ 4.5: 3374


In [49]:
gold_df = j2[j2["membership"] == "Gold"]


In [50]:
city_revenue_gold = gold_df.groupby("city")["total_amount"].sum().reset_index()
top_city = city_revenue_gold.loc[city_revenue_gold["total_amount"].idxmax(), "city"]


In [51]:
orders_in_top_city = len(gold_df[gold_df["city"] == top_city])
print("Top revenue city among Gold members:", top_city)
print("Number of orders placed there:", orders_in_top_city)


Top revenue city among Gold members: Chennai
Number of orders placed there: 1337
